In [1]:
import torch
import time
import keyboard
import pyautogui
from IPython.display import clear_output

from camera_wrapper import CameraWrapper
from hand_tracker import HandTracker
from online_k_means import OnlineKmeans
from live_dataset import LiveDataset
from autoencoder import HumanSignalAutoencoder
from ae_trainer import AETrainer
from hand_visualizer import HandVisualizer
from stats_visualizer import StatsVisualizer
from centroid_labeler import CentroidLabeler
from label_getter import LabelGetter

In [2]:
# settings

In [3]:
# architecture settings
human_signal_shape = (1, 21, 3) # 1 hand * 21 tracking points * 3 tracking dims
autoencoder_bottleneck = 42

# region classifier settings
okm_num_centroids = 6

# online k-means settings
okm_retrain_iters    = 15
okm_convergence_atol = 1e-6
okm_dwell_time       = 0.3   # seconds a centroid must be held before actuating

# live dataset settings
queue_size_per_class = 200

# autoenc training settings
lr = 0.0005
ae_batch_size = 128
ae_steps_per_frame = 5

# label settings — 'r' = rest, no action
label_keys = ['s', 'l', 'c', 'r']
num_labels = len(label_keys)

# label assignment confidence gate
label_margin_threshold = 0.75

# control keys
quit_key           = 'q'
tracker_toggle_key = 'space'

# minimum seconds between any two instagram actions
action_cooldown = 1.0

# instagram action coords
COORD_SCROLL_DOWN = (2475, 790)
COORD_LIKE        = (1639, 827)
COORD_COMMENTS    = (1643, 927)

action_coord_map = {
    's': COORD_SCROLL_DOWN,
    'l': COORD_LIKE,
    'c': COORD_COMMENTS,
    # 'r' intentionally absent — rest pose fires nothing
}

# torch settings
run_device = torch.device("cuda")

In [4]:
camera = CameraWrapper(camera_index=1)
hand_tracker = HandTracker("./hand_landmarker.task")

In [5]:
true_visualizer = HandVisualizer('True Hand')
decoded_visualizer = HandVisualizer('Decoded Hand')
stats_visualizer = StatsVisualizer()

In [6]:
autoenc = HumanSignalAutoencoder(human_signal_shape, autoencoder_bottleneck).to(run_device)
region_classifier = OnlineKmeans(okm_num_centroids, run_device, retrain_iters=okm_retrain_iters, convergence_atol=okm_convergence_atol)
live_dataset = LiveDataset(okm_num_centroids, queue_size_per_class)
ae_trainer = AETrainer(autoenc, lr=lr, device=run_device)
centroid_labeler = CentroidLabeler(okm_num_centroids, num_labels)
label_getter = LabelGetter(label_keys)

In [ ]:
start_time = time.time()
tracker_enabled = False
tracker_toggle_was_pressed = False
prev_class_idx = None
label_map = {i: None for i in range(okm_num_centroids)}
confidence_map = {i: 0.0 for i in range(okm_num_centroids)}
loss_val = float('nan')
_last_action_time = 0.0
_dwell_start = time.time()
_dwell_actuated = False

def ig_click(coord):
    global _last_action_time
    if time.time() - _last_action_time < action_cooldown:
        return
    pyautogui.click(*coord)
    _last_action_time = time.time()

while not keyboard.is_pressed(quit_key):

    # space: toggle tracker mode
    toggle_is_pressed = keyboard.is_pressed(tracker_toggle_key)
    if toggle_is_pressed and not tracker_toggle_was_pressed:
        tracker_enabled = not tracker_enabled
    tracker_toggle_was_pressed = toggle_is_pressed

    # capture frame
    img = camera()

    # track hand — skip frame if shape doesn't match (0 or 2+ hands)
    human_signal = hand_tracker(img, int((time.time() - start_time) * 1000)).to(run_device)
    if human_signal.shape != torch.Size(human_signal_shape):
        continue

    # encode
    with torch.no_grad():
        latent = autoenc.encode(human_signal.unsqueeze(0)).squeeze(0)

    # classify
    class_idx, dists = region_classifier.classify(latent)
    if class_idx is None:
        class_idx = 0

    # reset dwell timer on centroid change
    if class_idx != prev_class_idx:
        _dwell_start = time.time()
        _dwell_actuated = False

    # get ground truth label — s/l/c/r keys always active for labeling
    label = label_getter.get_label()
    label_key_pressed = label is not None
    if label is not None and dists is not None:
        sorted_dists = dists.sort().values
        margin = sorted_dists[0] / (sorted_dists[1] + 1e-8)
        if margin >= label_margin_threshold:
            label = None

    # store datapoint
    live_dataset.apply_datapoint(latent, human_signal, class_idx, label=label)

    # retrain autoencoder
    for _ in range(ae_steps_per_frame):
        batch = live_dataset.get_random_human_sigs(ae_batch_size)
        if batch is not None:
            loss_val = ae_trainer.train_step(batch)

    # retrain OKM
    all_latents = live_dataset.get_random_latents(live_dataset.total_size())
    region_classifier.retrain(all_latents)

    # recount labels when user is actively labeling
    if label_key_pressed:
        label_map = centroid_labeler.get_label_map(live_dataset)
        confidence_map = centroid_labeler.get_confidence_map(live_dataset)

    # tracker-actuated controls: fire after dwell, once per stable hold
    if tracker_enabled and not _dwell_actuated and (time.time() - _dwell_start) >= okm_dwell_time:
        label_idx = label_map.get(class_idx)
        if label_idx is not None:
            coord = action_coord_map.get(label_keys[label_idx])
            if coord is not None:
                ig_click(coord)
        _dwell_actuated = True

    prev_class_idx = class_idx

    # visualise
    true_visualizer.update(human_signal.detach().cpu())
    with torch.no_grad():
        decoded = autoenc(human_signal.unsqueeze(0)).squeeze(0)
    decoded_visualizer.update(decoded.detach().cpu())

    dist_val = dists[class_idx].item() if dists is not None else float('nan')
    label_display = ' | '.join(
        f"C{i}->{label_keys[label_map[i]] if label_map[i] is not None else '?'}"
        for i in range(okm_num_centroids)
    )
    mode_str = "TRACKER" if tracker_enabled else "MANUAL"
    stats_visualizer.update([
        f"AE stats: loss {loss_val:.6f} | batch size: {ae_batch_size}",
        f"LiveDataset stats: queues: {live_dataset.get_queue_sizes()}",
        f"OnlineKmeans stats: detected centroid: C{class_idx} (euclidean dist: {dist_val:.4f})",
        f"CentroidLabeler [{mode_str}]: held key: {label_keys[label] if label is not None else None} | labels: {label_display}",
    ])